In [0]:
# Databricks notebook source
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "churn_dev")
catalog, schema = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
model_name = f"{catalog}.{schema}.churn_model"

In [0]:
# COMMAND ----------
import mlflow, mlflow.sklearn
from mlflow.models import infer_signature
from mlflow import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

mlflow.set_registry_uri("databricks-uc")

pdf = spark.table(f"{catalog}.{schema}.gold_churn_features").toPandas()
cat = ["contract", "internet_service"]
num = ["tenure_months", "monthly_charges", "support_calls",
       "total_spend", "calls_per_year", "is_month_to_month"]
X, y = pdf[cat + num], pdf["churned"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipe = Pipeline([
    ("prep", ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat)],
                               remainder="passthrough")),
    ("clf", GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=42)),
])

with mlflow.start_run(run_name="gbm_churn") as run:
    pipe.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
    mlflow.log_metric("test_auc", auc)
    mlflow.log_param("n_estimators", 150)
    mlflow.sklearn.log_model(
        pipe, artifact_path="model",
        signature=infer_signature(X_tr, pipe.predict(X_tr)),
        registered_model_name=model_name)
print(f"AUC = {auc:.3f}")

In [0]:


# COMMAND ----------
import mlflow, mlflow.sklearn
from mlflow.models import infer_signature
from mlflow import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

mlflow.set_registry_uri("databricks-uc")

pdf = spark.table(f"{catalog}.{schema}.gold_churn_features").toPandas()
cat = ["contract", "internet_service"]
num = ["tenure_months", "monthly_charges", "support_calls",
       "total_spend", "calls_per_year", "is_month_to_month"]
X, y = pdf[cat + num], pdf["churned"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipe = Pipeline([
    ("prep", ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat)],
                               remainder="passthrough")),
    ("clf", GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=42)),
])

with mlflow.start_run(run_name="gbm_churn") as run:
    pipe.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
    mlflow.log_metric("test_auc", auc)
    mlflow.log_param("n_estimators", 150)
    mlflow.sklearn.log_model(
        pipe, artifact_path="model",
        signature=infer_signature(X_tr, pipe.predict(X_tr)),
        registered_model_name=model_name)
print(f"AUC = {auc:.3f}")


In [0]:

# COMMAND ----------
# Promotion gate: only move the 'champion' alias if the model clears a quality bar
client = MlflowClient()
latest = max(int(v.version) for v in client.search_model_versions(f"name='{model_name}'"))
if auc >= 0.70:
    client.set_registered_model_alias(model_name, "champion", latest)
    print(f"Version {latest} is now champion")
else:
    raise ValueError(f"AUC {auc:.3f} below threshold; not promoting")